# Notebook 03 — Baselines intra-dataset avec validation

**Datasets :** CIC-IDS2017 et UNSW-NB15.

les cinq splits `StratifiedGroupKFold` produisent des prédictions *out-of-fold* (OOF). Le seuil est sélectionné sur toutes ces prédictions, puis un modèle est entraîné sur 100 % du train externe avant l'évaluation sur le test externe.


## 1. Configuration
Exécuter les cellules dans l’ordre. La racine est détectée depuis le projet ou son dossier `notebooks/`.

In [1]:
from pathlib import Path
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, average_precision_score, confusion_matrix, precision_recall_curve)
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("default", category=ConvergenceWarning)
SEEDS = [42, 123, 2026]
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5
RUN_FULL_EXPERIMENT = True  # False pour exécuter uniquement le pilote
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT_DIR / "data" / "interim"
TABLES_DIR = ROOT_DIR / "results" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_COLUMNS = ["flow_duration_s", "fwd_packets", "bwd_packets", "fwd_bytes", "bwd_bytes"]
TARGET_COLUMN = "target"
RESULTS_FILE = TABLES_DIR / "intra_dataset_baselines_v2_oof.csv"
SUMMARY_FILE = TABLES_DIR / "intra_dataset_baselines_summary_v2_oof.csv"
print("Racine :", ROOT_DIR)
print("Données :", DATA_DIR)
print("Sortie :", RESULTS_FILE)
print("Seeds :", SEEDS)

Racine : D:\Master-SDIA\CYB-1-IDS-Generalization
Données : D:\Master-SDIA\CYB-1-IDS-Generalization\data\interim
Sortie : D:\Master-SDIA\CYB-1-IDS-Generalization\results\tables\intra_dataset_baselines_v2_oof.csv
Seeds : [42, 123, 2026]


## 2. Chargement des données et partitions externes
Les indices externes sont réutilisés à l’identique depuis le notebook d’harmonisation.

In [2]:
def load_dataset(dataset_name, sample_filename, splits_filename):
    sample_path = DATA_DIR / sample_filename
    splits_path = TABLES_DIR / splits_filename
    if not sample_path.exists(): raise FileNotFoundError(f"Échantillon introuvable : {sample_path}")
    if not splits_path.exists(): raise FileNotFoundError(f"Folds introuvables : {splits_path}")
    df = pd.read_csv(sample_path)
    required = FEATURE_COLUMNS + [TARGET_COLUMN]
    missing = [col for col in required if col not in df.columns]
    if missing: raise ValueError(f"{dataset_name}: colonnes manquantes {missing}")
    X = df[FEATURE_COLUMNS].copy()
    y = df[TARGET_COLUMN].astype(int).to_numpy()
    feature_index = pd.MultiIndex.from_frame(X)
    groups, _ = pd.factorize(feature_index, sort=False)
    groups = np.asarray(groups)
    with np.load(splits_path) as z:
        keys = [f"{p}_{i}" for i in range(1, N_OUTER_FOLDS+1) for p in ("train", "test")]
        missing_keys = [k for k in keys if k not in z.files]
        if missing_keys: raise KeyError(f"Clés absentes {missing_keys}; disponibles: {z.files}")
        outer_splits = [(z[f"train_{i}"].astype(int).copy(), z[f"test_{i}"].astype(int).copy()) for i in range(1, N_OUTER_FOLDS+1)]
    if set(np.unique(y)) != {0,1}: raise ValueError(f"{dataset_name}: target doit contenir 0 et 1")
    if not np.isfinite(X.to_numpy(dtype=float)).all(): raise ValueError(f"{dataset_name}: valeurs non finies")
    for fold, (tr, te) in enumerate(outer_splits, 1):
        if len(np.unique(tr)) != len(tr) or len(np.unique(te)) != len(te): raise AssertionError(f"Indices dupliqués fold {fold}")
        if not np.isin(tr, np.arange(len(X))).all() or not np.isin(te, np.arange(len(X))).all(): raise AssertionError(f"Indice hors limites fold {fold}")
        if np.intersect1d(tr, te).size: raise AssertionError(f"Chevauchement indices fold {fold}")
        if np.intersect1d(np.unique(groups[tr]), np.unique(groups[te])).size: raise AssertionError(f"Fuite groupes fold {fold}")
    print(f"\n{dataset_name}: X={X.shape}, groupes={len(np.unique(groups))}, classes={dict(zip(*np.unique(y, return_counts=True)))}")
    return {"name":dataset_name, "X":X, "y":y, "groups":groups, "outer_splits":outer_splits}

cic_data = load_dataset("CIC-IDS2017", "cic_harmonized_sample_v2.csv", "cic_group_splits_v2.npz")
unsw_data = load_dataset("UNSW-NB15", "unsw_harmonized_sample_v2.csv", "unsw_group_splits_v2.npz")


CIC-IDS2017: X=(100000, 5), groupes=74294, classes={np.int64(0): np.int64(80212), np.int64(1): np.int64(19788)}

UNSW-NB15: X=(100000, 5), groupes=64117, classes={np.int64(0): np.int64(87423), np.int64(1): np.int64(12577)}


## 3. Modèles
Le scaler est ajusté dans chaque pipeline uniquement sur le sous-ensemble d’entraînement correspondant.

In [3]:
def build_models(seed):
    return {
        "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=seed))]),
        "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample", random_state=seed, n_jobs=-1),
        "MLP": Pipeline([("scaler", StandardScaler()), ("model", MLPClassifier(hidden_layer_sizes=(64,32), activation="relu", solver="adam", max_iter=300, early_stopping=False, n_iter_no_change=15, random_state=seed))]),
    }
MODEL_NAMES = list(build_models(SEEDS[0]).keys())
print("Modèles :", MODEL_NAMES)

Modèles : ['Logistic Regression', 'Random Forest', 'MLP']


## 4. Sélection du seuil sur les prédictions OOF
La fonction aligne précision/rappel sur les seuils et maximise le F1. Elle refuse les entrées invalides plutôt que de choisir silencieusement un seuil de secours.

In [4]:
def select_threshold(y_true, probabilities):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    if len(y_true) != len(probabilities): raise ValueError("Tailles incompatibles")
    if not np.isfinite(probabilities).all(): raise ValueError("Probabilités non finies")
    if len(np.unique(y_true)) < 2: raise ValueError("Les deux classes doivent être présentes dans OOF")
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    if len(thresholds) == 0: raise ValueError("Aucun seuil disponible")
    den = precision[:-1] + recall[:-1]
    f1_values = np.divide(2*precision[:-1]*recall[:-1], den, out=np.zeros_like(den), where=den != 0)
    best = int(np.nanargmax(f1_values))
    return float(thresholds[best]), float(f1_values[best])

## 5. Création et contrôle des cinq splits internes
Chaque observation du train externe doit apparaître exactement une fois comme observation de validation OOF. Les groupes train/validation sont disjoints dans chaque split.

In [5]:
def build_and_validate_inner_splits(X_outer_train, y_outer_train, groups_outer_train, seed):
    cv = StratifiedGroupKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=seed)
    splits = list(cv.split(X_outer_train, y_outer_train, groups_outer_train))
    if len(splits) != N_INNER_FOLDS: raise AssertionError(f"{len(splits)} splits obtenus")
    validation_counts = np.zeros(len(X_outer_train), dtype=np.int8)
    for fold, (tr, va) in enumerate(splits, 1):
        if np.intersect1d(tr, va).size: raise AssertionError(f"Chevauchement indices split {fold}")
        if np.intersect1d(np.unique(groups_outer_train[tr]), np.unique(groups_outer_train[va])).size:
            raise AssertionError(f"Fuite groupes split {fold}")
        validation_counts[va] += 1
    if not np.all(validation_counts == 1):
        vals, counts = np.unique(validation_counts, return_counts=True)
        raise AssertionError(f"Couverture OOF incorrecte: {dict(zip(vals.tolist(), counts.tolist()))}")
    return splits
print("Contrôle des splits internes défini.")

Contrôle des splits internes défini.


## 6. Exécution d’une expérience OOF
Cinq modèles internes produisent les prédictions OOF. Le seuil est choisi sur toutes les observations du train externe. Un modèle neuf est ensuite entraîné sur tout ce train avant l’évaluation externe.

In [6]:
def run_one_experiment_v2(dataset, model_name, fold_number, seed):
    X, y, groups = dataset["X"], dataset["y"], dataset["groups"]
    outer_train_idx, outer_test_idx = dataset["outer_splits"][fold_number-1]
    X_outer_train = X.iloc[outer_train_idx]
    y_outer_train = y[outer_train_idx]
    groups_outer_train = groups[outer_train_idx]
    X_outer_test = X.iloc[outer_test_idx]
    y_outer_test = y[outer_test_idx]
    if np.intersect1d(np.unique(groups_outer_train), np.unique(groups[outer_test_idx])).size:
        raise AssertionError("Chevauchement groupes train/test externe")

    inner_splits = build_and_validate_inner_splits(X_outer_train, y_outer_train, groups_outer_train, seed)
    oof_probabilities = np.full(len(X_outer_train), np.nan, dtype=float)
    inner_training_time = 0.0
    for inner_fold, (inner_tr, inner_va) in enumerate(inner_splits, 1):
        model = build_models(seed)[model_name]
        start = time.perf_counter()
        model.fit(X_outer_train.iloc[inner_tr], y_outer_train[inner_tr])
        inner_training_time += time.perf_counter() - start
        oof_probabilities[inner_va] = model.predict_proba(X_outer_train.iloc[inner_va])[:,1]
    if not np.isfinite(oof_probabilities).all(): raise AssertionError("Prédictions OOF manquantes")
    threshold, oof_f1 = select_threshold(y_outer_train, oof_probabilities)

    final_model = build_models(seed)[model_name]
    start = time.perf_counter()
    final_model.fit(X_outer_train, y_outer_train)
    final_training_time = time.perf_counter() - start
    test_probabilities = final_model.predict_proba(X_outer_test)[:,1]
    test_predictions = (test_probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_outer_test, test_predictions, labels=[0,1]).ravel()
    return {
        "dataset":dataset["name"], "model":model_name, "fold":fold_number, "seed":seed,
        "n_train_outer":len(X_outer_train), "n_oof_validation":len(oof_probabilities), "n_test":len(X_outer_test),
        "n_inner_folds":len(inner_splits), "threshold":threshold, "oof_validation_f1":oof_f1,
        "test_f1":f1_score(y_outer_test,test_predictions,zero_division=0),
        "test_pr_auc":average_precision_score(y_outer_test,test_probabilities),
        "test_precision":precision_score(y_outer_test,test_predictions,zero_division=0),
        "test_recall":recall_score(y_outer_test,test_predictions,zero_division=0),
        "test_accuracy":accuracy_score(y_outer_test,test_predictions),
        "test_balanced_accuracy":balanced_accuracy_score(y_outer_test,test_predictions),
        "inner_training_time_s":inner_training_time, "final_training_time_s":final_training_time,
        "total_training_time_s":inner_training_time+final_training_time,
        "tn":int(tn), "fp":int(fp), "fn":int(fn), "tp":int(tp),
    }
print("run_one_experiment_v2 définie.")

run_one_experiment_v2 définie.


## 7. Essai pilote — CIC-IDS2017, Logistic Regression, fold 1, seed 42
Cette expérience comprend cinq entraînements internes et un entraînement final. Vérifier son résultat avant de poursuivre.

In [7]:
pilot_result_v2 = run_one_experiment_v2(cic_data, "Logistic Regression", 1, 42)
display(pd.DataFrame([pilot_result_v2]).T.rename(columns={0:"Valeur"}))
print("Matrice [[TN, FP], [FN, TP]]:")
print(np.array([[pilot_result_v2["tn"], pilot_result_v2["fp"]], [pilot_result_v2["fn"], pilot_result_v2["tp"]]]))

,Valeur
dataset,CIC-IDS2017
model,Logistic Regression
fold,1
seed,42
n_train_outer,79999
n_oof_validation,79999
n_test,20001
n_inner_folds,5
threshold,0.473062
oof_validation_f1,0.581226


Matrice [[TN, FP], [FN, TP]]:
[[11679  4364]
 [  578  3380]]


## 8. Expérience complète avec sauvegarde progressive
90 expériences externes sont prévues. Chaque expérience comprend cinq entraînements internes et un entraînement final, soit jusqu’à 540 ajustements de modèles. La sauvegarde progressive permet de reprendre après interruption. Le fichier v1 n’est jamais modifié.

In [8]:
if RESULTS_FILE.exists():
    results_df = pd.read_csv(RESULTS_FILE)
    completed = set(zip(results_df.dataset, results_df.model, results_df.fold.astype(int), results_df.seed.astype(int)))
    print(f"Reprise : {len(completed)} expériences déjà enregistrées.")
else:
    results_df = pd.DataFrame()
    completed = set()

if RUN_FULL_EXPERIMENT:
    datasets_to_run = [cic_data, unsw_data]
    total = len(datasets_to_run)*len(MODEL_NAMES)*N_OUTER_FOLDS*len(SEEDS)
    print(f"Expériences terminées : {len(completed)}/{total}")
    for dataset in datasets_to_run:
        for model_name in MODEL_NAMES:
            for fold_number in range(1,N_OUTER_FOLDS+1):
                for seed in SEEDS:
                    key = (dataset["name"],model_name,fold_number,seed)
                    if key in completed: continue
                    print(f"{dataset['name']} | {model_name} | fold={fold_number} | seed={seed}")
                    result = run_one_experiment_v2(dataset,model_name,fold_number,seed)
                    results_df = pd.concat([results_df,pd.DataFrame([result])],ignore_index=True)
                    results_df.to_csv(RESULTS_FILE,index=False)
                    completed.add(key)
    print("Expérience terminée :", RESULTS_FILE)
else:
    print("RUN_FULL_EXPERIMENT=False : pilote uniquement.")

Reprise : 34 expériences déjà enregistrées.
Expériences terminées : 34/90
CIC-IDS2017 | MLP | fold=2 | seed=123
CIC-IDS2017 | MLP | fold=2 | seed=2026
CIC-IDS2017 | MLP | fold=3 | seed=42
CIC-IDS2017 | MLP | fold=3 | seed=123


D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


CIC-IDS2017 | MLP | fold=3 | seed=2026


D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


CIC-IDS2017 | MLP | fold=4 | seed=42


D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  war

... (7 outputs left out)

D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


CIC-IDS2017 | MLP | fold=5 | seed=2026


D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  war

UNSW-NB15 | Logistic Regression | fold=1 | seed=42
UNSW-NB15 | Logistic Regression | fold=1 | seed=123
UNSW-NB15 | Logistic Regression | fold=1 | seed=2026
UNSW-NB15 | Logistic Regression | fold=2 | seed=42
UNSW-NB15 | Logistic Regression | fold=2 | seed=123
UNSW-NB15 | Logistic Regression | fold=2 | seed=2026
UNSW-NB15 | Logistic Regression | fold=3 | seed=42
UNSW-NB15 | Logistic Regression | fold=3 | seed=123
UNSW-NB15 | Logistic Regression | fold=3 | seed=2026
UNSW-NB15 | Logistic Regression | fold=4 | seed=42
UNSW-NB15 | Logistic Regression | fold=4 | seed=123
UNSW-NB15 | Logistic Regression | fold=4 | seed=2026
UNSW-NB15 | Logistic Regression | fold=5 | seed=42
UNSW-NB15 | Logistic Regression | fold=5 | seed=123
UNSW-NB15 | Logistic Regression | fold=5 | seed=2026
UNSW-NB15 | Random Forest | fold=1 | seed=42
UNSW-NB15 | Random Forest | fold=1 | seed=123
UNSW-NB15 | Random Forest | fold=1 | seed=2026
UNSW-NB15 | Random Forest | fold=2 | seed=42
UNSW-NB15 | Random Forest | fold=2 | 

## 9. Résultats détaillés

In [9]:
if RESULTS_FILE.exists():
    results_df = pd.read_csv(RESULTS_FILE)
    display(results_df.sort_values(["dataset","model","fold","seed"]).reset_index(drop=True))
else: print("Aucun résultat v2 : exécuter la cellule précédente.")

,dataset,model,fold,seed,n_train_outer,n_oof_validation,n_test,n_inner_folds,threshold,oof_validation_f1,...,test_recall,test_accuracy,test_balanced_accuracy,inner_training_time_s,final_training_time_s,total_training_time_s,tn,fp,fn,tp
0,CIC-IDS2017,Logistic Regression,1,42,79999,79999,20001,5,0.473062,0.581226,...,0.853967,0.752912,0.790974,1.601638,0.302217,1.903855,11679,4364,578,3380
1,CIC-IDS2017,Logistic Regression,1,123,79999,79999,20001,5,0.473691,0.584138,...,0.848914,0.784661,0.808861,1.401940,0.318939,1.720879,12334,3709,598,3360
2,CIC-IDS2017,Logistic Regression,1,2026,79999,79999,20001,5,0.472414,0.579627,...,0.868621,0.751962,0.795901,0.898796,0.180157,1.078953,11602,4441,520,3438
3,CIC-IDS2017,Logistic Regression,2,42,80001,80001,19999,5,0.469221,0.572523,...,0.886783,0.758988,0.807124,0.805757,0.235882,1.041639,11670,4372,448,3509
4,CIC-IDS2017,Logistic Regression,2,123,80001,80001,19999,5,0.472220,0.582836,...,0.878191,0.817391,0.840292,0.787351,0.200433,0.987783,12872,3170,482,3475
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,UNSW-NB15,Random Forest,4,123,79991,79991,20009,5,0.447747,0.956801,...,0.958333,0.987955,0.975279,27.984800,7.862389,35.847189,17353,136,105,2415
86,UNSW-NB15,Random Forest,4,2026,79991,79991,20009,5,0.365208,0.956931,...,0.971825,0.988705,0.981481,27.471643,7.951166,35.422809,17334,155,71,2449
87,UNSW-NB15,Random Forest,5,42,80005,80005,19995,5,0.335000,0.955189,...,0.972941,0.989497,0.982409,28.220026,8.090039,36.310065,17340,142,68,2445
88,UNSW-NB15,Random Forest,5,123,80005,80005,19995,5,0.405625,0.956385,...,0.961799,0.988447,0.977038,28.062659,7.916149,35.978808,17347,135,96,2417


## 10. Agrégation moyenne ± écart-type
L’écart-type décrit la variabilité observée ; ce n’est pas un intervalle de confiance, car les folds ne constituent pas des répétitions totalement indépendantes.

In [10]:
if RESULTS_FILE.exists():
    results_df = pd.read_csv(RESULTS_FILE)
    metrics = ["test_f1","test_pr_auc","test_precision","test_recall","test_accuracy","test_balanced_accuracy","threshold","oof_validation_f1","total_training_time_s"]
    summary = results_df.groupby(["dataset","model"])[metrics].agg(["mean","std"]).round(4)
    display(summary)
    summary.to_csv(SUMMARY_FILE)
    print("Synthèse sauvegardée :", SUMMARY_FILE)
else: print("Exécuter d’abord l’expérience complète.")

test_f1         test_pr_auc          \
                                   mean     std        mean     std   
dataset     model                                                     
CIC-IDS2017 Logistic Regression  0.5982  0.0252      0.5252  0.0207   
            MLP                  0.7126  0.0397      0.8840  0.0228   
            Random Forest        0.9540  0.0227      0.9829  0.0094   
UNSW-NB15   Logistic Regression  0.6375  0.0445      0.4063  0.0917   
            MLP                  0.8625  0.0955      0.9189  0.0448   
            Random Forest        0.9544  0.0073      0.9814  0.0112   

                                test_precision         test_recall          \
                                          mean     std        mean     std   
dataset     model                                                            
CIC-IDS2017 Logistic Regression         0.4633  0.0244      0.8454  0.0398   
            MLP                         0.8409  0.1666      0.6743  0.1801   
            Random Forest               0.9398  0.0448      0.9698  0.0058   
UNSW-NB15   Logistic Regression         0.5198  0.0274      0.8254  0.0832   
            MLP                         0.8142  0.1402      0.9363  0.0691   
            Random Forest               0.9405  0.0059      0.9689  0.0166   

                                test_accuracy         test_balanced_accuracy  \
                                         mean     std                   mean   
dataset     model                                                              
CIC-IDS2017 Logistic Regression        0.7750  0.0182                 0.8015   
            MLP                        0.8944  0.0258                 0.8115   
            Random Forest              0.9813  0.0098                 0.9770   
UNSW-NB15   Logistic Regression        0.8825  0.0120                 0.8581   
            MLP                        0.9594  0.0348                 0.9495   
            Random Forest              0.9884  0.0018                 0.9801   

                                        threshold         oof_validation_f1  \
                                    std      mean     std              mean   
dataset     model                                                             
CIC-IDS2017 Logistic Regression  0.0206    0.4727  0.0014            0.5826   
            MLP                  0.0599    0.3825  0.0715            0.7568   
            Random Forest        0.0059    0.6739  0.0253            0.9540   
UNSW-NB15   Logistic Regression  0.0423    0.6910  0.0178            0.6410   
            MLP                  0.0378    0.4002  0.0739            0.9095   
            Random Forest        0.0081    0.3910  0.0422            0.9558   

                                        total_training_time_s            
                                    std                  mean       std  
dataset     model                                                        
CIC-IDS2017 Logistic Regression  0.0154                1.1813    0.4308  
            MLP                  0.0174             1013.8033  249.7932  
            Random Forest        0.0059               35.0563    2.6925  
UNSW-NB15   Logistic Regression  0.0115                1.1723    0.2067  
            MLP                  0.0334              563.3541  177.9091  
            Random Forest        0.0023               36.6350    1.5897

Synthèse sauvegardée : D:\Master-SDIA\CYB-1-IDS-Generalization\results\tables\intra_dataset_baselines_summary_v2_oof.csv


## 11. Vérifications de complétude

In [11]:
if RESULTS_FILE.exists():
    results_df = pd.read_csv(RESULTS_FILE)
    expected = {(d,m,f,s) for d in ["CIC-IDS2017","UNSW-NB15"] for m in MODEL_NAMES for f in range(1,N_OUTER_FOLDS+1) for s in SEEDS}
    observed = set(zip(results_df.dataset,results_df.model,results_df.fold.astype(int),results_df.seed.astype(int)))
    missing = sorted(expected-observed)
    duplicates = int(results_df.duplicated(subset=["dataset","model","fold","seed"]).sum())
    print("Attendus :",len(expected)," | obtenus :",len(observed)," | manquants :",len(missing)," | doublons :",duplicates)
    if missing: print("Premières combinaisons manquantes :",missing[:10])
    if duplicates: raise AssertionError("Combinaisons expérimentales dupliquées")
else: print("Fichier de résultats absent.")

Attendus : 90  | obtenus : 90  | manquants : 0  | doublons : 0


## 12. Interprétation et limites

1. Le F1 dépend du seuil ; Average Precision résume la courbe précision-rappel sur plusieurs seuils.
2. Le seuil est sélectionné uniquement sur les prédictions OOF du train externe ; le test externe n’intervient jamais dans ce choix.
3. Le modèle final est ajusté sur tout le train externe, alors que les modèles internes qui produisent OOF sont ajustés sur des sous-ensembles. Cette différence est inhérente à cette procédure et doit être signalée.
4. Les écarts ne peuvent donc pas être attribués uniquement au seuil OOF.
5. Les avertissements de convergence du MLP doivent être examinés ; `max_iter=300` ne garantit pas la convergence.
6. Les groupes sont des doublons exacts sur cinq caractéristiques, pas des sessions réseau.
7. Les résultats concernent les échantillons harmonisés de 100 000 observations.
8. Les baselines intra-dataset ne démontrent pas la généralisation cross-dataset.

**Fichiers produits :**
- `results/tables/intra_dataset_baselines_v2_oof.csv`
- `results/tables/intra_dataset_baselines_summary_v2_oof.csv`